# W4Q1 — Email Campaign Quasi-Experiment (May 2022)
**Question:** Did users who received an email campaign in May 2022 convert at higher rates than those who did not?

**Audience:** Marketing Team  
**Data source:** `ANALYTICS.MARTS.MART_EMAIL_ENGAGEMENT` + `MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W4Q1_email_quasi_experiment.sql`

---

> ⚠️ **Causal inference caveat:** This is an observational quasi-experiment, not a randomised trial.
> Treatment (receiving the email) may correlate with pre-existing user characteristics.
> Any observed difference in conversion should be interpreted with caution.
> Report effect sizes alongside statistical significance.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W4Q1_email_quasi_experiment.sql') as f:
    sql = f.read()

df = query(sql)
print(f"Rows: {len(df):,}")
print(f"Treatment (received May email): {df['received_may_email'].sum():,}")
print(f"Control (did not receive):      {(~df['received_may_email']).sum():,}")
print(f"\nCampaigns in May 2022: {df[df['received_may_email']]['may_campaign'].unique()}")

## Conversion Rates — Treatment vs Control

In [ ]:
outcomes = {
    'has_trial': 'Trial Rate',
    'has_3m_subscription': '3m Rate',
    'has_12m_subscription': '12m Rate',
}

rows = []
for col, label in outcomes.items():
    for received in [True, False]:
        grp = df[df['received_may_email'] == received]
        n = len(grp)
        converted = grp[col].sum()
        rows.append({
            'Group': 'Treatment (received)' if received else 'Control (not received)',
            'Outcome': label,
            'N': n,
            'Converted': converted,
            'Rate (%)': round(converted / n * 100, 1),
        })

comparison = pd.DataFrame(rows)
display(comparison)

In [ ]:
pivot = comparison.pivot(index='Outcome', columns='Group', values='Rate (%)')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, color=PALETTE[:2], edgecolor='white')
ax.set_title('Conversion Rates — May 2022 Email Recipients vs Non-Recipients', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Group')
sns.despine()
plt.tight_layout()
plt.savefig('../output/W4Q1_email_experiment_conversion.png', bbox_inches='tight')
plt.show()

## Chi-Square Test — Statistical Significance

In [ ]:
for col, label in outcomes.items():
    treated     = df[df['received_may_email'] == True]
    control     = df[df['received_may_email'] == False]
    contingency = [
        [treated[col].sum(),  len(treated)  - treated[col].sum()],
        [control[col].sum(),  len(control)  - control[col].sum()],
    ]
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    print(f"{label:20s}  chi2={chi2:.2f}  p={p:.4f}  {'✓ significant' if p < 0.05 else '— not significant'}")

## Findings

- **May 2022 campaigns identified:** ...
- **Treatment vs control group sizes:** ...
- **Trial conversion rate difference:** ...
- **3m conversion rate difference:** ...
- **12m conversion rate difference:** ...
- **Statistical significance:** ...
- **Causal interpretation caveat:** ...
- **Recommendation:** ...